# 📊 12｜共享單車綜合專題：清理決定跟著問題走，最後自己提問

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/12_bike_capstone/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：把 01～11 學過的清理技巧整合起來，並學會一件事：**清理決定要跟著問題走**——同一筆問題資料，對某個問題要排除，對另一個問題卻可以保留。最後由你自己提出問題。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：讀入三個檔案、完成清理決策表，最後用**一張垂直長條圖**回答「一天中哪個時段借車最多？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出清理決策表、各小時借車數與清理後資料。
3. **選做挑戰（單元五～七）**：比較平日與週末、合併天氣檔比較晴雨，最後**自己提出一個問題**完成專題。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/bike_trips.csv`（騎乘紀錄）、`data/stations.csv`（站點）、`data/weather_daily.csv`（每日天氣）。請先閱讀[本例資料卡](./data/DATA_CARD.md)。**騎乘檔的一列代表：一次借還車紀錄**（2025-04-01～06-30）；站點檔一列是一個站，天氣檔一列是一天。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`騎乘編號`（可檢查重複）、`借車站`／`還車站`（對照站點的鑰匙）。
- 🔵 **條件 (Dimension)**：`借車時間`／`還車時間`（時間）、`使用者類型`、`年齡層`（類別）；天氣檔的 `天氣`。
- 🟢 **指標 (Metric)**：衍生的 `騎乘分鐘`；站點的 `車位數`；天氣的 `最高溫`（°C）、`降雨量`（毫米）。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：一天中哪個時段借車最多？
- 📅 **比較＋分母（選做）**：平日與週末，每天各時段的平均借車數有何不同？
- 🌧️ **比較＋合併（選做）**：晴天、多雲、雨天的每日平均騎乘次數差多少？
- 🙋 **自行提問（選做）**：用五步 SOP 提出並回答你自己的問題。

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **先去重複**：20 筆完全重複的紀錄要刪除，以下所有筆數都以去重複後為準。
- ⚠️ **時間填反**：25 筆還車時間早於借車時間；本例判斷為兩欄填反，**對調並記錄**，而不是刪除。
- ⚠️ **清理決定跟著問題走**：8 筆騎乘超過 2 天（忘了還車）——算「騎乘時間」時要排除，算「借車時段」時可以保留；60 筆還車站空白——只影響「還車站」相關分析。
- ⚠️ **比較要用相同分母**：平日天數比週末多、雨天和晴天的天數也不同；比較時要用「每日平均」，不是總次數。
- ⚠️ **天氣不是唯一原因**：雨天騎乘較少，但雨天也可能剛好落在平日；只能說「同時出現」，不能直接說「下雨造成減少多少」。

### 4. 決定最適圖表
- **24 個小時比借車數** ➔ **垂直長條圖**（小時依 0→23 排列；標出早晚兩個尖峰）。
- **平日 vs 週末各小時平均（選做）** ➔ **折線圖**（兩條線，X 軸為小時；比較的是每日平均）。
- **三種天氣的每日平均騎乘（選做）** ➔ **垂直長條圖**（標出各天氣的天數 n）。
- **自行提問（選做）** ➔ 依你的問題類型，用「看懂資料與選對圖表」的規則自己決定。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'bike_trips.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/12_bike_capstone/data/bike_trips.csv',
    'stations.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/12_bike_capstone/data/stations.csv',
    'weather_daily.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/12_bike_capstone/data/weather_daily.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取三個檔案，盤點資料問題（貼標籤）

> 🎯 **思維重點**：先盤點，不急著清：重複幾筆？時間填反幾筆？超長騎乘幾筆？缺值在哪裡？每個檔案一列代表什麼？

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/ 中的 bike_trips.csv、stations.csv、weather_daily.csv：
1. 每個檔案顯示前 3 列、列數、欄位名稱與缺值數，並說明「一列代表什麼」。
2. 在騎乘資料中檢查：完全重複的列有幾筆？「騎乘編號」重複幾筆？
3. 把借車時間、還車時間轉成日期時間格式，計算騎乘分鐘；統計騎乘分鐘小於 0 與超過 2 天（2,880 分鐘）的筆數。
4. 用文字回報欄位角色（標識、條件、指標），並列出你發現的資料問題清單。
先不要修改資料，也不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

trips = pd.read_csv('data/bike_trips.csv')
stations = pd.read_csv('data/stations.csv')
weather = pd.read_csv('data/weather_daily.csv')

for name, table, unit in [('騎乘紀錄', trips, '一次借還車'), ('站點', stations, '一個站點'), ('每日天氣', weather, '一天')]:
    print(f"\n=== {name}：{len(table)} 列，一列代表「{unit}」 ===")
    display(table.head(3))
    print('缺值數：', table.isna().sum()[lambda s: s > 0].to_dict() or '無')

print(f"\n完全重複的列：{trips.duplicated().sum()}；騎乘編號重複：{trips['騎乘編號'].duplicated().sum()}")
minutes = (pd.to_datetime(trips['還車時間']) - pd.to_datetime(trips['借車時間'])).dt.total_seconds() / 60
print(f"騎乘分鐘 < 0（時間可能填反）：{(minutes < 0).sum()} 筆；超過 2 天：{(minutes > 2 * 24 * 60).sum()} 筆")
print("⚠️ 以上是「尚未去除重複」的數字；重複列裡也可能有問題資料，去重複後要重新清點。")

### 📝 單元二【必做】：邊界檢核：做出清理決策表，決定跟著問題走

> 🎯 **思維重點**：不是「全部清掉」：每個問題都寫下筆數、處理方式、會影響哪些分析。重複要刪、填反要對調、超長騎乘與還車站空白先「標記」，等到需要的問題再排除。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請依下列規則清理騎乘資料（不要修改原始檔），並做一張「清理決策表」：
1. 刪除完全重複的列，印出處理前後筆數。
2. 還車時間早於借車時間的紀錄，判斷為兩欄填反：把兩個時間對調，並新增「時間已對調」欄位標記。
3. 重新計算騎乘分鐘；超過 2 天（2,880 分鐘）的紀錄不刪除，新增「超長騎乘」欄位標記。
4. 還車站空白的紀錄不刪除，保留缺值。
5. 從借車時間拆出「借車小時」與「借車日期」。
6. 清理決策表欄位：資料問題、筆數、處理方式、影響的分析、不影響的分析。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
rides = trips.drop_duplicates().copy()
print(f"① 刪除完全重複：{len(trips)} → {len(rides)} 筆")

rides['借車時間'] = pd.to_datetime(rides['借車時間'])
rides['還車時間'] = pd.to_datetime(rides['還車時間'])
swapped = rides['還車時間'] < rides['借車時間']
rides.loc[swapped, ['借車時間', '還車時間']] = rides.loc[swapped, ['還車時間', '借車時間']].to_numpy()
rides['時間已對調'] = swapped
print(f"② 時間填反並對調：{swapped.sum()} 筆")

rides['騎乘分鐘'] = (rides['還車時間'] - rides['借車時間']).dt.total_seconds() / 60
rides['超長騎乘'] = rides['騎乘分鐘'] > 2 * 24 * 60
print(f"③ 超長騎乘（> 2 天）標記：{rides['超長騎乘'].sum()} 筆；騎乘分鐘最小值 {rides['騎乘分鐘'].min():.1f}")
print(f"④ 還車站空白（保留）：{rides['還車站'].isna().sum()} 筆")

rides['借車小時'] = rides['借車時間'].dt.hour
rides['借車日期'] = rides['借車時間'].dt.normalize()

decisions = pd.DataFrame([
    ('完全重複的紀錄', len(trips) - len(rides), '刪除', '所有分析', '—'),
    ('還車時間早於借車時間', int(swapped.sum()), '判斷為填反，對調並標記', '騎乘時間、借車時段', '—'),
    ('騎乘超過 2 天', int(rides['超長騎乘'].sum()), '保留並標記', '騎乘時間（需排除）', '借車時段、借車次數'),
    ('還車站空白', int(rides['還車站'].isna().sum()), '保留缺值', '還車站相關分析（需排除）', '借車時段、借車次數'),
], columns=['資料問題', '筆數', '處理方式', '影響的分析', '不影響的分析'])
display(decisions)

### 📝 單元三【必做】：提問積木【比較】— 一天中哪個時段借車最多？（垂直長條圖）

> 🎯 **思維重點**：24 個小時依時間順序排列，用【垂直長條圖】；借車時段不受「超長騎乘」影響，這 8 筆可以保留。早、晚各標出一個尖峰。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用清理後的資料，畫出各「借車小時」借車次數的垂直長條圖：
1. 先產出 0～23 點的借車次數表（沒有借車的小時也顯示 0）。
2. X 軸為借車小時（0～23，依時間順序），Y 軸為借車次數，從 0 開始。
3. 分別找出上午（0～11 點）與下午（12～23 點）借車最多的小時，用醒目顏色標示並標註次數。
4. 標題寫出資料期間與筆數，圖表文字全部使用繁體中文。
5. 在註解說明：為什麼這個問題不需要排除超長騎乘與還車站空白的紀錄？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

# 借車時段只用到「借車時間」：超長騎乘與還車站空白都不影響，所以全部 6,000 筆都保留
hourly = rides['借車小時'].value_counts().reindex(range(24), fill_value=0)
am_peak, pm_peak = hourly[:12].idxmax(), hourly[12:].idxmax()
colors = ['#e74c3c' if h in (am_peak, pm_peak) else '#3498db' for h in hourly.index]

plt.figure(figsize=(11, 5))
plt.bar(hourly.index, hourly.values, color=colors)
for h in (am_peak, pm_peak):
    plt.text(h, hourly[h] + 8, f'{h} 點\n{hourly[h]} 次', ha='center', fontweight='bold')
plt.xticks(range(24))
plt.ylim(0, hourly.max() * 1.2)
plt.title(f'各小時借車次數（2025/4/1～6/30，n = {len(rides):,}）', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('借車時間（點）')
plt.ylabel('借車次數（次）')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()
print(f"💡 早上尖峰 {am_peak} 點（{hourly[am_peak]} 次）、傍晚尖峰 {pm_peak} 點（{hourly[pm_peak]} 次）。")

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能說出三個檔案各自「一列代表什麼」，以及騎乘資料的四種問題（重複、時間填反、超長騎乘、還車站空白）。
- [ ] 能用清理決策表說明每個問題的筆數、處理方式，以及它會影響哪些分析、不影響哪些分析。
- [ ] 能從長條圖與表格指出早晚兩個借車尖峰，並核對數字一致。
- 💬 **討論題**：為什麼超長騎乘的 8 筆，在這張圖裡可以保留？什麼樣的問題就必須排除它們？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 原始 6,020 筆，刪除 20 筆完全重複後剩 **6,000 筆**；25 筆時間填反已對調、8 筆超過 2 天已標記、60 筆還車站空白。
- 借車時段長條圖呈現**早上 8 點（535 次）**與**傍晚 17 點（633 次）**兩個尖峰。

</details>

若今天繼續做報表，請進行**單元四**；單元五～七是選做挑戰，單元七請你自己提出問題。

---

### 📝 單元四【報表應用】：匯出清理決策表、各小時借車數與清理後資料

> 🎯 **思維重點**：專題報告要讓別人能重現：清理決策表放在第一個工作表，接著是結果與清理後資料。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將專題成果匯出成 Excel：
1. 存為 output/共享單車專題.xlsx（若 output 目錄不存在請自動建立）。
2. 工作表依序為：「清理決策表」「各小時借車數」「清理後騎乘」（含時間已對調、超長騎乘、騎乘分鐘、借車小時等欄位）。
3. 匯出後重新讀取，核對清理後騎乘為 6,000 筆、各小時借車數合計也是 6,000。
4. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
os.makedirs('output', exist_ok=True)
output_path = 'output/共享單車專題.xlsx'
with pd.ExcelWriter(output_path) as writer:
    decisions.to_excel(writer, sheet_name='清理決策表', index=False)
    hourly.rename_axis('借車小時').rename('借車次數').reset_index().to_excel(writer, sheet_name='各小時借車數', index=False)
    rides.drop(columns='借車日期').to_excel(writer, sheet_name='清理後騎乘', index=False)

check_rides = pd.read_excel(output_path, sheet_name='清理後騎乘')
check_hourly = pd.read_excel(output_path, sheet_name='各小時借車數')
print(f"重新讀取：清理後騎乘 {len(check_rides)} 筆；各小時借車數合計 {check_hourly['借車次數'].sum()}")

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 檔案！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較＋分母】— 平日與週末的借車時段（折線圖）

> 🎯 **思維重點**：平日有 65 天、週末只有 26 天，直接比總次數不公平；改成「每日平均借車數」（分母＝天數），再用兩條折線比較形狀。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請比較平日與週末各小時的借車情形：
1. 依借車日期判斷平日（週一～週五）或週末（週六、週日）。
2. 用天氣檔的日期當作完整日曆，計算平日與週末各有幾天。
3. 計算「各小時每日平均借車數」＝該類日子在該小時的借車次數 ÷ 該類日子的天數。
4. 畫折線圖：X 軸為 0～23 點，兩條線分別代表平日與週末，Y 軸從 0 開始，圖例寫出天數。
5. 用文字描述兩條線的形狀差異（例如通勤雙峰 vs 午後單峰）。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
calendar = pd.to_datetime(weather['日期'])
day_type = lambda d: d.dt.day_of_week.map(lambda x: '週末' if x >= 5 else '平日')
n_days = day_type(calendar).value_counts()
print("天數：", n_days.to_dict())

rides['日別'] = day_type(rides['借車日期'])
per_day = (rides.groupby(['日別', '借車小時']).size().unstack(0).reindex(range(24), fill_value=0)
           .div(n_days, axis=1))

plt.figure(figsize=(10, 5))
for kind, color in [('平日', '#3498db'), ('週末', '#e67e22')]:
    plt.plot(per_day.index, per_day[kind], marker='o', color=color, linewidth=2, label=f'{kind}（{n_days[kind]} 天）')
plt.xticks(range(24))
plt.ylim(0, per_day.max().max() * 1.15)
plt.title('平日與週末：各小時每日平均借車數', fontsize=13, fontweight='bold')
plt.xlabel('借車時間（點）')
plt.ylabel('每日平均借車數（次／天）')
plt.legend()
plt.grid(linestyle='--', alpha=0.4)
plt.show()
print("💡 平日有早晚通勤雙峰；週末是午後單峰。比較的是「每日平均」，所以天數不同也公平。")

### 📝 單元六【選做挑戰】：提問積木【比較＋合併】— 晴天、多雲、雨天的每日平均騎乘（垂直長條圖）

> 🎯 **思維重點**：天氣在另一個檔案，要先用「日期」合併；三種天氣的天數不同，比較的是每日平均。再檢查各天氣中週末的比例，避免把週末效果誤當成天氣效果。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請合併天氣資料，比較不同天氣的騎乘量：
1. 依借車日期計算每日騎乘次數，與 weather_daily.csv 用日期合併（以天氣檔為主，沒有騎乘的日子算 0）；印出合併前後天數。
2. 依天氣計算：天數、每日平均騎乘次數、其中週末天數的比例。
3. 畫垂直長條圖：X 軸為天氣（晴、多雲、雨），Y 軸為每日平均騎乘次數，從 0 開始，柱子上標註平均值與天數 n。
4. 用文字說明雨天平均少了多少，並提醒：天氣不是唯一因素，不能直接說「下雨造成減少 X%」。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
daily = rides.groupby('借車日期').size().rename('騎乘次數')
weather['日期'] = pd.to_datetime(weather['日期'])
day_weather = weather.merge(daily, left_on='日期', right_index=True, how='left').fillna({'騎乘次數': 0})
print(f"天氣檔 {len(weather)} 天；合併後 {len(day_weather)} 天；有騎乘紀錄的天數 {daily.size}")

day_weather['週末'] = day_weather['日期'].dt.day_of_week >= 5
weather_stats = day_weather.groupby('天氣').agg(天數=('日期', 'count'), 每日平均騎乘=('騎乘次數', 'mean'),
                                              週末比例=('週末', 'mean')).reindex(['晴', '多雲', '雨'])
display(weather_stats.style.format({'每日平均騎乘': '{:.1f}', '週末比例': '{:.0%}'}))

plt.figure(figsize=(7, 5))
bars = plt.bar(weather_stats.index, weather_stats['每日平均騎乘'], color=['#f4d03f', '#aab7b8', '#5dade2'], width=0.5)
for bar, n in zip(bars, weather_stats['天數']):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f'{bar.get_height():.1f}\n(n={n} 天)', ha='center')
plt.ylim(0, weather_stats['每日平均騎乘'].max() * 1.25)
plt.title('不同天氣的每日平均騎乘次數', fontsize=13, fontweight='bold')
plt.xlabel('天氣')
plt.ylabel('每日平均騎乘次數（次／天）')
plt.show()
print("⚠️ 邊界提醒：各天氣的週末比例不同，天氣不是唯一因素；只能說雨天騎乘較少，不能直接算出下雨的影響大小。")

### 📝 單元七【選做挑戰】：自行提問：用五步 SOP 回答你自己的問題

> 🎯 **思維重點**：換你當分析師：先填好「提問卡」，再請 AI 執行。下方參考程式示範一個例子：「會員與單次使用者，每次騎多久？」——這個問題就**必須排除**超長騎乘。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請先在下方填好提問卡，再把整段貼給 AI：
【提問卡】
- 我的問題：＿＿＿＿＿＿（例如：會員與單次使用者，每次騎乘時間有什麼不同？）
- 問題類型：比較／組成／分布／關係／趨勢（選一個）
- 需要的欄位與角色：＿＿＿＿＿＿
- 需要排除或特別處理的資料（參考清理決策表）：＿＿＿＿＿＿
- 選擇的圖表與理由：＿＿＿＿＿＿
- 這份資料「不能」回答的延伸推論：＿＿＿＿＿＿

請依提問卡使用清理後的 rides 資料：先產出回答問題的數字表，再畫圖；圖表使用繁體中文並標示單位與樣本數；最後用三句話說明：結論、資料限制、不能推論的事。
```

> 💡 下方參考程式是「會員 vs 單次使用者的騎乘時間」範例。你的問題不同時，請用自己的提問卡請 AI 產生程式，並對照提問卡逐項驗收。

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
# 範例提問：會員與單次使用者，每次騎乘時間有什麼不同？（分布＋比較 → 箱形圖）
# 騎乘時間會被「超長騎乘」嚴重拉高，所以這個問題必須排除那 8 筆
valid = rides[~rides['超長騎乘']]
print(f"排除超長騎乘：{len(rides)} → {len(valid)} 筆")
duration = valid.groupby('使用者類型')['騎乘分鐘'].agg(筆數='count', 平均='mean', 中位數='median').round(1)
display(duration)

plt.figure(figsize=(7, 5))
plt.boxplot([valid.loc[valid['使用者類型'] == t, '騎乘分鐘'] for t in duration.index], showfliers=False,
            patch_artist=True, boxprops=dict(facecolor='#a8dadc', color='#1d3557'), medianprops=dict(color='#e74c3c', linewidth=2))
plt.xticks([1, 2], [f'{t}（n={n:,}）' for t, n in duration['筆數'].items()])
plt.title('會員與單次使用者的騎乘時間（已排除超過 2 天的紀錄）', fontsize=12, fontweight='bold')
plt.ylabel('騎乘分鐘（分鐘）')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()
print(f"💡 單次使用者騎乘時間中位數 {duration.loc['單次', '中位數']} 分鐘，會員 {duration.loc['會員', '中位數']} 分鐘。")
print("⚠️ 限制：資料沒有騎乘目的與距離，不能推論單次使用者「比較愛騎車」。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：6,020 → 6,000 筆；清理決策表完整；借車尖峰為早上 8 點與傍晚 17 點。
- [ ] **報表應用**：Excel 含清理決策表、各小時借車數與清理後資料，筆數可核對。
- [ ] **選做挑戰**：平日／週末、晴雨比較都使用每日平均；自行提問有提問卡、數字表、圖與三句結論。
- [ ] **AI 協作**：能說明自己提出的問題、自己做的清理決定、AI 幫忙執行的事，以及如何確認結果。